In [29]:
!pip install numpy==1.26.4

In [30]:
!pip install surprise

In [31]:
!wget 'https://drive.google.com/uc?id=1m0rwReR09achL0xTM6QPoN4tykz5bOMx' -O MovieLens.zip

--2025-12-22 18:56:49--  https://drive.google.com/uc?id=1m0rwReR09achL0xTM6QPoN4tykz5bOMx
Resolving drive.google.com (drive.google.com)... 142.251.2.138, 142.251.2.101, 142.251.2.102, ...
Connecting to drive.google.com (drive.google.com)|142.251.2.138|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1m0rwReR09achL0xTM6QPoN4tykz5bOMx [following]
--2025-12-22 18:56:49--  https://drive.usercontent.google.com/download?id=1m0rwReR09achL0xTM6QPoN4tykz5bOMx
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 142.251.2.132, 2607:f8b0:4023:c0d::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|142.251.2.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 847695 (828K) [application/octet-stream]
Saving to: ‘MovieLens.zip’

MovieLens.zip       100%[===================>] 827.83K  --.-KB/s    in 0.1s    

2025-12-22 18:56:51 (7.17 MB/s) - ‘Movi

In [32]:
!unzip MovieLens.zip

Archive:  MovieLens.zip
replace links.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: links.csv               
replace movies.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: movies.csv              
replace ratings.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: ratings.csv             
replace tags.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: tags.csv                


In [61]:
import pandas as pd
import numpy as np

movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')

movies.shape, ratings.shape

((9742, 3), (100836, 4))

### Гибридная рекомендательная система

**Blending + Switching**:

* **Content-based (CB):** TF-IDF по `title + genres`, скор = cosine similarity к профилю пользователя (средний вектор понравившихся фильмов).
* **Collaborative (CF):** SVD из `surprise`, скор = предсказанный рейтинг.
* **Hybrid (blending):** итоговый скор = `alpha * CF + (1 - alpha) * CB`.
* **Switching:** `alpha` зависит от количества оценок пользователя в train (cold-start -> меньше alpha).

In [62]:
movies = movies.copy()
ratings = ratings.copy()

movies['title'] = movies['title'].fillna('').astype('string')
movies['genres'] = movies['genres'].fillna('').astype('string')

movies['text'] = (
    movies['title'].str.replace(r'[\(\)\d]', ' ', regex=True).str.lower()
    + ' '
    + movies['genres'].str.replace('|', ' ', regex=False).str.lower()
)

ratings['userId'] = ratings['userId'].astype('string')
ratings['movieId'] = ratings['movieId'].astype('string')
ratings['timestamp'] = pd.to_numeric(ratings['timestamp'], errors='coerce')

movies[['movieId', 'title', 'genres', 'text']].head()

,movieId,title,genres,text
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,toy story adventure animation children ...
1,2,Jumanji (1995),Adventure|Children|Fantasy,jumanji adventure children fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance,grumpier old men comedy romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,waiting to exhale comedy drama romance
4,5,Father of the Bride Part II (1995),Comedy,father of the bride part ii comedy


In [63]:
ratings = ratings.dropna(subset=['timestamp']).copy()
ratings['timestamp'] = ratings['timestamp'].astype('int64')

idx_test = (
    ratings.sort_values(['userId', 'timestamp'])
    .groupby('userId')
    .tail(1)
    .index
)

test = ratings.loc[idx_test].copy()
train = ratings.drop(idx_test).copy()

train.shape, test.shape, train['userId'].nunique(), test['userId'].nunique()

((100226, 4), (610, 4), 610, 610)

In [64]:
movies = movies.copy()
movies['movieId'] = movies['movieId'].astype('string')

item_ids_all = movies['movieId'].to_numpy(dtype='object')
itemid2row = pd.Series(np.arange(len(item_ids_all), dtype='int32'), index=item_ids_all)

tfidf = TfidfVectorizer(min_df=2, max_features=20000)
X_item = tfidf.fit_transform(movies['text'].to_list())

X_item.shape, item_ids_all[:3]

((9742, 3198), array(['1', '2', '3'], dtype=object))

In [65]:
u = eval_users[0]
cand = get_candidates(u)[:20]
sum([1 for mid in cand if mid in itemid2row.index]), len(cand), cand[:5]

(20, 20, ['318', '589', '150', '4993', '858'])

In [66]:
K = 10
rel_threshold = 4.0

train_items_by_user = train.groupby('userId')['movieId'].apply(set)
test_rel_by_user = (
    test[test['rating'] >= rel_threshold]
    .groupby('userId')['movieId']
    .apply(set)
)

eval_users = test_rel_by_user.index.intersection(train_items_by_user.index)
eval_users = [str(u) for u in eval_users]

len(eval_users)

363

In [67]:
def precision_at_k(recommended, relevant, k):
    rec_k = recommended[:k]
    if len(rec_k) == 0:
        return 0.0
    return float(len([x for x in rec_k if x in relevant]) / k)

def recall_at_k(recommended, relevant, k):
    if len(relevant) == 0:
        return 0.0
    rec_k = recommended[:k]
    return float(len([x for x in rec_k if x in relevant]) / len(relevant))

def apk(recommended, relevant, k):
    rec_k = recommended[:k]
    hit_count = 0
    score = 0.0
    for i, item in enumerate(rec_k, start=1):
        if item in relevant:
            hit_count += 1
            score += hit_count / i
    if len(relevant) == 0:
        return 0.0
    return float(score / min(len(relevant), k))

def mapk(recs_by_user, rel_by_user, k):
    return float(np.mean([apk(recs_by_user[u], rel_by_user[u], k) for u in recs_by_user]))

In [68]:
M = 5000

popularity = train.groupby('movieId')['userId'].size().sort_values(ascending=False)
topM_popular = popularity.index.to_list()[:M]

def get_candidates(user_id):
    seen = train_items_by_user.get(user_id, set())
    return [mid for mid in topM_popular if mid not in seen]

In [69]:
def recommend_popular(user_id, k):
    cand = get_candidates(user_id)
    return cand[:k]

In [70]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(min_df=2, max_features=20000)
X_item = tfidf.fit_transform(movies['text'].to_list())

item_ids_all = movies['movieId'].to_numpy(dtype='object')
itemid2row = pd.Series(np.arange(len(item_ids_all), dtype='int32'), index=item_ids_all)

liked_train = (
    train[train['rating'] >= rel_threshold]
    .groupby('userId')['movieId']
    .apply(list)
)

def cb_scores_for_candidates(user_id, candidates):
    liked = liked_train.get(user_id, [])
    liked = [i for i in liked if i in itemid2row.index]
    if len(liked) == 0:
        return np.zeros(len(candidates), dtype='float32')
    rows = itemid2row.loc[liked].to_numpy(dtype='int32')
    profile = X_item[rows].mean(axis=0)
    profile = np.asarray(profile).ravel()
    sim_all = cosine_similarity(X_item, profile.reshape(1, -1)).ravel().astype('float32')
    return np.array([sim_all[itemid2row[mid]] if mid in itemid2row.index else 0.0 for mid in candidates], dtype='float32')

def recommend_content(user_id, k):
    cand = get_candidates(user_id)
    if len(cand) == 0:
        return []
    scores = cb_scores_for_candidates(user_id, cand)
    order = np.argpartition(-scores, min(k, len(scores) - 1))[:k]
    order = order[np.argsort(-scores[order])]
    return [cand[i] for i in order]

In [71]:
from surprise import Dataset, Reader, SVD

reader = Reader(rating_scale=(float(train['rating'].min()), float(train['rating'].max())))
data_train = Dataset.load_from_df(train[['userId', 'movieId', 'rating']], reader)
trainset = data_train.build_full_trainset()

algo_svd = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
algo_svd.fit(trainset)

In [72]:
def cf_scores_for_candidates(user_id, candidates):
    return np.array([algo_svd.predict(user_id, mid).est for mid in candidates], dtype='float32')

def recommend_svd(user_id, k):
    cand = get_candidates(user_id)
    if len(cand) == 0:
        return []
    scores = cf_scores_for_candidates(user_id, cand)
    order = np.argpartition(-scores, min(k, len(scores) - 1))[:k]
    order = order[np.argsort(-scores[order])]
    return [cand[i] for i in order]

In [73]:
def minmax_norm(x):
    x = np.asarray(x, dtype='float32')
    mn = float(np.min(x))
    mx = float(np.max(x))
    if mx - mn < 1e-8:
        return np.zeros_like(x)
    return (x - mn) / (mx - mn)

def recommend_hybrid_alpha(user_id, k, alpha):
    cand = get_candidates(user_id)
    if len(cand) == 0:
        return []
    cf = cf_scores_for_candidates(user_id, cand)
    cb = cb_scores_for_candidates(user_id, cand)
    cf = minmax_norm(cf)
    cb = minmax_norm(cb)
    scores = float(alpha) * cf + (1.0 - float(alpha)) * cb
    order = np.argpartition(-scores, min(k, len(scores) - 1))[:k]
    order = order[np.argsort(-scores[order])]
    return [cand[i] for i in order]

In [74]:
rng = np.random.default_rng(42)

eval_users_sub = eval_users
if len(eval_users_sub) > 2000:
    eval_users_sub = rng.choice(eval_users_sub, size=2000, replace=False).tolist()

rel_eval = {u: test_rel_by_user[u] for u in eval_users_sub if u in test_rel_by_user.index}

alphas = np.linspace(0, 1, 11)
rows = []

for a in alphas:
    recs = {}
    for u in rel_eval.keys():
        recs[u] = recommend_hybrid_alpha(u, K, float(a))
    rows.append({'alpha': float(a), 'map@k': mapk(recs, rel_eval, K)})

df_alpha = pd.DataFrame(rows).sort_values('map@k', ascending=False).reset_index(drop=True)
best_alpha = float(df_alpha.iloc[0]['alpha'])

df_alpha, best_alpha

(    alpha     map@k
 0     0.9  0.008474
 1     0.8  0.007545
 2     1.0  0.007143
 3     0.7  0.005441
 4     0.6  0.002972
 5     0.3  0.002617
 6     0.4  0.002388
 7     0.2  0.002181
 8     0.5  0.002158
 9     0.0  0.001705
 10    0.1  0.001607,
 0.9)

In [75]:
def eval_recommender(recommender_fn):
    recs = {u: recommender_fn(u) for u in rel_eval.keys()}
    p = float(np.mean([precision_at_k(recs[u], rel_eval[u], K) for u in recs]))
    r = float(np.mean([recall_at_k(recs[u], rel_eval[u], K) for u in recs]))
    m = mapk(recs, rel_eval, K)
    return p, r, m

p_pop, r_pop, m_pop = eval_recommender(lambda u: recommend_popular(u, K))
p_cb, r_cb, m_cb = eval_recommender(lambda u: recommend_content(u, K))
p_svd, r_svd, m_svd = eval_recommender(lambda u: recommend_svd(u, K))
p_h, r_h, m_h = eval_recommender(lambda u: recommend_hybrid_alpha(u, K, best_alpha))

df_results = pd.DataFrame([
    {'model': 'Popular(topM)', 'precision@k': p_pop, 'recall@k': r_pop, 'map@k': m_pop},
    {'model': 'Content-based', 'precision@k': p_cb, 'recall@k': r_cb, 'map@k': m_cb},
    {'model': 'SVD', 'precision@k': p_svd, 'recall@k': r_svd, 'map@k': m_svd},
    {'model': f'Hybrid(alpha={best_alpha:.2f})', 'precision@k': p_h, 'recall@k': r_h, 'map@k': m_h},
]).sort_values('map@k', ascending=False).reset_index(drop=True)

df_results

,model,precision@k,recall@k,map@k
0,Popular(topM),0.006061,0.060606,0.019490
1,Hybrid(alpha=0.90),0.002204,0.022039,0.008474
2,SVD,0.001377,0.013774,0.007143
3,Content-based,0.000826,0.008264,0.001705


In [76]:
example_user = rel_eval.keys().__iter__().__next__()

hist = train[train['userId'] == example_user].merge(
    movies[['movieId', 'title', 'genres']],
    on='movieId',
    how='left'
).sort_values('rating', ascending=False).head(10)

hist

,userId,movieId,rating,timestamp,title,genres
115,1,1954,5.0,964982176,Rocky (1976),Drama
80,1,1226,5.0,964983618,"Quiet Man, The (1952)",Drama|Romance
127,1,2078,5.0,964982838,"Jungle Book, The (1967)",Animation|Children|Comedy|Musical
128,1,2090,5.0,964982838,"Rescuers, The (1977)",Adventure|Animation|Children|Crime|Drama
130,1,2094,5.0,964982653,"Rocketeer, The (1991)",Action|Adventure|Sci-Fi
134,1,2115,5.0,964982529,Indiana Jones and the Temple of Doom (1984),Action|Adventure|Fantasy
135,1,2116,5.0,964982876,"Lord of the Rings, The (1978)",Adventure|Animation|Children|Fantasy
136,1,2137,5.0,964982791,Charlotte's Web (1973),Animation|Children
137,1,2139,5.0,964982791,"Secret of NIMH, The (1982)",Adventure|Animation|Children|Drama
138,1,2141,5.0,964982838,"American Tail, An (1986)",Adventure|Animation|Children|Comedy


In [77]:
recs = recommend_hybrid_alpha(example_user, K, best_alpha)

pd.DataFrame({'movieId': recs}).merge(
    movies[['movieId', 'title', 'genres']],
    on='movieId',
    how='left'
)

,movieId,title,genres
0,48516,"Departed, The (2006)",Crime|Drama|Thriller
1,69481,"Hurt Locker, The (2008)",Action|Drama|Thriller|War
2,106642,"Day of the Doctor, The (2013)",Adventure|Drama|Sci-Fi
3,60069,WALL·E (2008),Adventure|Animation|Children|Romance|Sci-Fi
4,3030,Yojimbo (1961),Action|Adventure
5,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX
6,122926,Untitled Spider-Man Reboot (2017),Action|Adventure|Fantasy
7,912,Casablanca (1942),Drama|Romance
8,1223,"Grand Day Out with Wallace and Gromit, A (1989)",Adventure|Animation|Children|Comedy|Sci-Fi
9,1204,Lawrence of Arabia (1962),Adventure|Drama|War
